# NORTHSTAR -- Tower 15: App Makers QA for Solace Browser

**Tower:** 15 -- Tower of App Makers
**DNA:** `platform(thriving) = sdk(intuitive) x docs(complete) x store(fair) x feedback(fast) x templates(inspiring) x monetization(transparent) x community(growing)`
**Target:** solace-browser at `/home/phuc/projects/solace-browser/`
**Floors probed:** 1-12 (First-Time Builder, Recipe Author, OAuth3 Integrator, Store Publisher, Revenue Developer, API Consumer, Theme Designer, Extension Builder, Documentation Writer, Community Contributor, Accessibility App Maker, i18n App Maker)
**Protocol:** FORECAST -> CROSS-REF -> AUDIT -> FIX -> VERIFY -> SEAL -> POSTMORTEM
**Total Questions:** 343 (49 floors x 7 questions)
**Auth:** 65537

In [ ]:
# Cell 1: Bootstrap — imports, paths, helper functions
# Tower 15: App Makers QA — probing solace-browser for app maker experience
import os, sys, re, json, hashlib, glob as globmod
from pathlib import Path
from datetime import datetime

NORTHSTAR = "app-makers-qa"
NOTEBOOK_PATH = "notebooks/qa/05-app-makers-qa.ipynb"
TOWER = 15
TOWER_NAME = "Tower of App Makers"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER_ROOT / "web"
SRC = BROWSER_ROOT / "src"
DATA = BROWSER_ROOT / "data" / "default"
CSS_FILE = WEB / "css" / "site.css"
JS_DIR = WEB / "js"
DOCS_DIR = WEB / "docs"
RECIPES_DIR = DATA / "recipes"
LOCALES_DIR = BROWSER_ROOT / "app" / "locales"

results = []

def record(probe_id, name, passed, detail=""):
    """Record a PASS/FAIL result with evidence detail."""
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    """Safely read a file, returning empty string if missing."""
    p = Path(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""

def file_exists(path):
    """Check if file exists at path."""
    return Path(path).exists()

def count_files(directory, pattern="*"):
    """Count files matching glob pattern in directory."""
    d = Path(directory)
    if not d.exists():
        return 0
    return len(list(d.glob(pattern)))

def search_in_files(directory, pattern, file_glob="*.html"):
    """Search for regex pattern in files matching glob. Return list of (filepath, match_count)."""
    hits = []
    d = Path(directory)
    if not d.exists():
        return hits
    for f in d.rglob(file_glob):
        content = read_file(f)
        matches = re.findall(pattern, content, re.IGNORECASE)
        if matches:
            hits.append((str(f.relative_to(BROWSER_ROOT)), len(matches)))
    return hits

assert BROWSER_ROOT.exists(), f"solace-browser not found at {BROWSER_ROOT}"
print(f"Tower {TOWER}: {TOWER_NAME}")
print(f"Browser root: {BROWSER_ROOT}")
print(f"Web dir:      {WEB} (exists={WEB.exists()})")
print(f"Recipes dir:  {RECIPES_DIR} (exists={RECIPES_DIR.exists()})")
print(f"Docs dir:     {DOCS_DIR} (exists={DOCS_DIR.exists()})")
print(f"Locales dir:  {LOCALES_DIR} (exists={LOCALES_DIR.exists()})")
print("Bootstrap complete.")

In [ ]:
# ============================================================
# Floor 1: First-Time App Builder
# Persona: someone who has never built a Solace app before
# Questions probed:
#   Q1: Does /create-app page exist?
#   Q2: Does create-app explain what an app IS in plain language?
#   Q3: Is there a create-app wizard/funnel (describe -> preview -> test -> submit)?
#   Q4: Are there example apps or templates a beginner can clone?
#   Q5: Does the documentation explain apps vs recipes vs skills?
# ============================================================

# Q1: Does /create-app page exist?
create_app_path = WEB / "create-app.html"
record("F1-Q1", "Create-app page exists (/create-app.html)", file_exists(create_app_path),
       f"Path: {create_app_path}")

# Q2: Does create-app explain what an app IS?
if create_app_path.exists():
    ca_content = read_file(create_app_path)
    # Look for plain-language explanation of what an app is
    has_explanation = bool(re.search(
        r'what.{0,20}(app|automat|recipe)|describe.{0,30}(want|idea|automat)|plain.?language',
        ca_content, re.IGNORECASE
    ))
    # Look for hero/intro section explaining the concept
    has_hero = bool(re.search(r'ca-hero|hero|eyebrow', ca_content, re.IGNORECASE))
    record("F1-Q2", "Create-app page explains concept in plain language",
           has_explanation or has_hero,
           f"explanation={has_explanation}, hero_section={has_hero}")
else:
    record("F1-Q2", "Create-app page explains concept", False, "Page does not exist")

# Q3: Create-app wizard funnel (describe -> preview -> test -> submit)
if create_app_path.exists():
    ca_content = read_file(create_app_path)
    has_describe = bool(re.search(r'describe|idea|what.*automat', ca_content, re.IGNORECASE))
    has_preview = bool(re.search(r'preview', ca_content, re.IGNORECASE))
    has_test = bool(re.search(r'test|try|sandbox', ca_content, re.IGNORECASE))
    has_submit = bool(re.search(r'submit|publish|store', ca_content, re.IGNORECASE))
    funnel_steps = sum([has_describe, has_preview, has_test, has_submit])
    record("F1-Q3", "Create-app has wizard funnel (describe/preview/test/submit)",
           funnel_steps >= 3,
           f"describe={has_describe}, preview={has_preview}, test={has_test}, submit={has_submit} ({funnel_steps}/4)")
else:
    record("F1-Q3", "Create-app wizard funnel", False, "Page does not exist")

# Q4: Example apps or templates for beginners
example_recipes = list(RECIPES_DIR.rglob("*.json")) if RECIPES_DIR.exists() else []
example_recipe_count = len(example_recipes)
# Check if create-app references examples
has_examples_ref = False
if create_app_path.exists():
    ca_content = read_file(create_app_path)
    has_examples_ref = bool(re.search(r'example|template|sample|clone|starter', ca_content, re.IGNORECASE))
record("F1-Q4", "Example apps/templates available for beginners",
       example_recipe_count > 5 and has_examples_ref,
       f"{example_recipe_count} recipes exist, create-app references examples={has_examples_ref}")

# Q5: Documentation explains apps vs recipes vs skills
docs_explain = False
for doc_file in [WEB / "docs.html", DOCS_DIR / "quick-start.html"]:
    if doc_file.exists():
        doc_content = read_file(doc_file)
        has_app = bool(re.search(r'\bapp\b', doc_content, re.IGNORECASE))
        has_recipe = bool(re.search(r'\brecipe\b', doc_content, re.IGNORECASE))
        has_skill = bool(re.search(r'\bskill\b', doc_content, re.IGNORECASE))
        if has_app and has_recipe:
            docs_explain = True
record("F1-Q5", "Docs explain difference between apps, recipes, skills", docs_explain)

In [ ]:
# ============================================================
# Floor 2: Recipe Author
# Persona: developer creating and testing recipes
# Questions probed:
#   Q1: Is there a recipe JSON schema or format documentation?
#   Q2: Do existing recipes follow a consistent schema (id, steps, oauth3_scopes)?
#   Q3: Can a recipe author see all available step types?
#   Q4: Is recipe versioning supported (version field in recipes)?
#   Q5: Are there validation tools that catch recipe errors?
# ============================================================

# Q1: Recipe format documented anywhere?
recipe_doc_found = False
recipe_doc_locations = []

# Check docs pages for recipe documentation
for doc_path in [WEB / "docs.html", DOCS_DIR / "quick-start.html", DOCS_DIR / "mcp.html"]:
    if doc_path.exists():
        content = read_file(doc_path)
        if re.search(r'recipe.{0,50}(format|schema|structure|json|steps)', content, re.IGNORECASE):
            recipe_doc_found = True
            recipe_doc_locations.append(str(doc_path.relative_to(BROWSER_ROOT)))

# Check CONTRIBUTING.md
contributing = BROWSER_ROOT / "CONTRIBUTING.md"
if contributing.exists():
    content = read_file(contributing)
    if re.search(r'recipe', content, re.IGNORECASE):
        recipe_doc_found = True
        recipe_doc_locations.append("CONTRIBUTING.md")

record("F2-Q1", "Recipe format is documented", recipe_doc_found,
       f"Found in: {recipe_doc_locations}" if recipe_doc_locations else "No recipe format docs found")

# Q2: Do existing recipes follow a consistent schema?
recipes = list(RECIPES_DIR.rglob("*.json")) if RECIPES_DIR.exists() else []
schema_fields = {"id": 0, "steps": 0, "oauth3_scopes": 0, "version": 0, "description": 0, "platform": 0}
valid_recipes = 0
for rp in recipes[:20]:  # Sample first 20
    try:
        data = json.loads(read_file(rp))
        if isinstance(data, dict):
            for field in schema_fields:
                if field in data:
                    schema_fields[field] += 1
            if "id" in data or "steps" in data:
                valid_recipes += 1
    except json.JSONDecodeError:
        pass

sampled = min(len(recipes), 20)
record("F2-Q2", "Recipes follow consistent schema",
       valid_recipes >= sampled * 0.7 if sampled > 0 else False,
       f"{valid_recipes}/{sampled} valid, fields: {schema_fields}")

# Q3: Available step types documented
step_types_found = set()
for rp in recipes[:30]:
    try:
        data = json.loads(read_file(rp))
        if isinstance(data, dict) and "steps" in data:
            for step in data["steps"]:
                if isinstance(step, dict) and "action" in step:
                    step_types_found.add(step["action"])
    except (json.JSONDecodeError, KeyError):
        pass

record("F2-Q3", "Multiple recipe step types exist in codebase",
       len(step_types_found) >= 3,
       f"{len(step_types_found)} step types: {sorted(step_types_found)[:15]}")

# Q4: Recipe versioning supported
versioned_count = schema_fields.get("version", 0)
record("F2-Q4", "Recipes support versioning (version field)",
       versioned_count >= sampled * 0.5 if sampled > 0 else False,
       f"{versioned_count}/{sampled} recipes have version field")

# Q5: Recipe validation tools
validator_hits = search_in_files(SRC, r'recipe.*valid|validate.*recipe|RecipeValidator', "*.py")
validator_hits += search_in_files(BROWSER_ROOT / "scripts", r'recipe.*valid|validate.*recipe', "*.py")
# Also check for JSON schema files
schema_files = list(BROWSER_ROOT.rglob("*recipe*schema*")) + list(BROWSER_ROOT.rglob("*schema*recipe*"))
record("F2-Q5", "Recipe validation tools exist",
       len(validator_hits) > 0 or len(schema_files) > 0,
       f"Validators: {validator_hits}, Schema files: {[str(s) for s in schema_files[:5]]}")

In [ ]:
# ============================================================
# Floor 3: OAuth3 Integrator
# Persona: developer integrating OAuth3 scopes into their app
# Questions probed:
#   Q1: Does /docs/oauth3 page exist with scope documentation?
#   Q2: Are OAuth3 scopes listed with human-readable descriptions?
#   Q3: Does the app-detail page show which scopes an app requests?
#   Q4: Are there code examples for OAuth3 token handling?
#   Q5: Does the OAuth3 consent dialog show app maker identity?
# ============================================================

# Q1: OAuth3 docs page exists
oauth3_doc = DOCS_DIR / "oauth3.html"
record("F3-Q1", "OAuth3 documentation page exists (/docs/oauth3.html)",
       file_exists(oauth3_doc),
       f"Path: {oauth3_doc}")

# Q2: OAuth3 scopes documented with descriptions
if oauth3_doc.exists():
    oauth3_content = read_file(oauth3_doc)
    # Look for scope listings
    has_scope_list = bool(re.search(r'scope', oauth3_content, re.IGNORECASE))
    has_descriptions = bool(re.search(r'(read|write|execute|admin).*scope|scope.*description',
                                       oauth3_content, re.IGNORECASE))
    # Check for structured scope table or list
    has_scope_table = bool(re.search(r'<table|<li.*scope|scope.*<li', oauth3_content, re.IGNORECASE))
    record("F3-Q2", "OAuth3 scopes documented with human-readable descriptions",
           has_scope_list and (has_descriptions or has_scope_table),
           f"scope_mentioned={has_scope_list}, descriptions={has_descriptions}, table={has_scope_table}")
else:
    record("F3-Q2", "OAuth3 scopes documented", False, "OAuth3 page does not exist")

# Q3: App-detail page shows requested scopes
app_detail = WEB / "app-detail.html"
if app_detail.exists():
    ad_content = read_file(app_detail)
    shows_scopes = bool(re.search(r'scope|permission|oauth', ad_content, re.IGNORECASE))
    record("F3-Q3", "App-detail page shows OAuth3 scopes/permissions",
           shows_scopes,
           f"References scopes/permissions={shows_scopes}")
else:
    record("F3-Q3", "App-detail page shows scopes", False, "app-detail.html does not exist")

# Q4: Code examples for OAuth3 token handling
oauth3_code_examples = False
if oauth3_doc.exists():
    content = read_file(oauth3_doc)
    has_code = bool(re.search(r'<code|<pre|```', content))
    has_token = bool(re.search(r'token|refresh|revok', content, re.IGNORECASE))
    oauth3_code_examples = has_code and has_token
record("F3-Q4", "OAuth3 docs include code examples for token handling",
       oauth3_code_examples,
       f"code_blocks={has_code if oauth3_doc.exists() else False}, token_refs={has_token if oauth3_doc.exists() else False}")

# Q5: OAuth3 consent dialog exists
oauth3_confirm_js = JS_DIR / "yinyang-oauth3-confirm.js"
if oauth3_confirm_js.exists():
    confirm_content = read_file(oauth3_confirm_js)
    shows_identity = bool(re.search(r'app.*name|identity|publisher|author|developer', confirm_content, re.IGNORECASE))
    shows_scopes = bool(re.search(r'scope|permission', confirm_content, re.IGNORECASE))
    record("F3-Q5", "OAuth3 consent dialog shows app identity and scopes",
           shows_identity or shows_scopes,
           f"identity={shows_identity}, scopes={shows_scopes}")
else:
    record("F3-Q5", "OAuth3 consent dialog exists", False, "yinyang-oauth3-confirm.js not found")

In [ ]:
# ============================================================
# Floor 4: Store Publisher
# Persona: developer submitting apps to the Solace store
# Questions probed:
#   Q1: Does /app-store page exist?
#   Q2: Does app-store have categories and search?
#   Q3: Does app-store show clear acceptance criteria?
#   Q4: Is the rung-gating system explained?
#   Q5: Is the sealed store policy explained (users suggest, we implement)?
# ============================================================

# Q1: App store page exists
app_store_path = WEB / "app-store.html"
record("F4-Q1", "App store page exists (/app-store.html)",
       file_exists(app_store_path),
       f"Path: {app_store_path}")

# Q2: App store has categories and search
if app_store_path.exists():
    as_content = read_file(app_store_path)
    has_categories = bool(re.search(r'categor|communication|productiv|sales|engineer|social',
                                      as_content, re.IGNORECASE))
    has_search = bool(re.search(r'search|filter|<input.*search', as_content, re.IGNORECASE))
    record("F4-Q2", "App store has categories and search",
           has_categories and has_search,
           f"categories={has_categories}, search={has_search}")
else:
    record("F4-Q2", "App store has categories and search", False, "Page does not exist")

# Q3: Acceptance criteria documented
acceptance_found = False
for path in [app_store_path, WEB / "docs.html", BROWSER_ROOT / "CONTRIBUTING.md"]:
    if path.exists():
        content = read_file(path)
        if re.search(r'accept.*criteria|review.*process|submission.*guideline|rejection|quality.*standard',
                     content, re.IGNORECASE):
            acceptance_found = True
            break
record("F4-Q3", "App submission acceptance criteria documented", acceptance_found)

# Q4: Rung-gating system explained
rung_explained = False
rung_locations = []
for path in [app_store_path, WEB / "docs.html", DOCS_DIR / "quick-start.html",
             BROWSER_ROOT / "CONTRIBUTING.md"]:
    if path.exists():
        content = read_file(path)
        if re.search(r'rung|trust.?level|verification.*ladder', content, re.IGNORECASE):
            rung_explained = True
            rung_locations.append(str(path.relative_to(BROWSER_ROOT)))
record("F4-Q4", "Rung-gating system explained to publishers",
       rung_explained,
       f"Found in: {rung_locations}" if rung_locations else "Not documented")

# Q5: Sealed store policy explained
sealed_explained = False
for path in [app_store_path, WEB / "docs.html", BROWSER_ROOT / "CONTRIBUTING.md"]:
    if path.exists():
        content = read_file(path)
        if re.search(r'sealed|users?\s+suggest|we\s+implement|no\s+plugin', content, re.IGNORECASE):
            sealed_explained = True
            break
record("F4-Q5", "Sealed store policy explained", sealed_explained)

In [ ]:
# ============================================================
# Floor 5: Revenue-Seeking Developer
# Persona: developer wanting to monetize apps on the platform
# Questions probed:
#   Q1: Is the revenue split documented?
#   Q2: Are pricing/monetization options visible on app-store or create-app?
#   Q3: Is there a pricing page explaining developer earnings?
#   Q4: Are Stripe/payment integration references present?
#   Q5: Can developers set app prices (price field in app manifest)?
# ============================================================

# Q1: Revenue split documented
revenue_doc_found = False
revenue_locations = []
search_paths = [
    WEB / "app-store.html",
    WEB / "create-app.html",
    WEB / "docs.html",
    BROWSER_ROOT / "CONTRIBUTING.md",
]
for path in search_paths:
    if path.exists():
        content = read_file(path)
        if re.search(r'revenue.{0,30}split|earn|royalt|developer.{0,20}(share|cut|percent)',
                     content, re.IGNORECASE):
            revenue_doc_found = True
            revenue_locations.append(str(path.relative_to(BROWSER_ROOT)))
record("F5-Q1", "Revenue split documented for developers",
       revenue_doc_found,
       f"Found in: {revenue_locations}" if revenue_locations else "No revenue docs")

# Q2: Monetization options visible in app-store or create-app
monetization_visible = False
for page in [WEB / "app-store.html", WEB / "create-app.html"]:
    if page.exists():
        content = read_file(page)
        if re.search(r'price|paid|free|premium|monetiz|earn', content, re.IGNORECASE):
            monetization_visible = True
            break
record("F5-Q2", "Monetization options visible in store UI",
       monetization_visible)

# Q3: Pricing page that explains developer earnings
pricing_pages = [WEB / "pricing.html", WEB / "home.html"]
earnings_explained = False
for pp in pricing_pages:
    if pp.exists():
        content = read_file(pp)
        if re.search(r'developer|maker|creator|publish.*earn|app.*price',
                     content, re.IGNORECASE):
            earnings_explained = True
            break
record("F5-Q3", "Pricing page explains developer earnings",
       earnings_explained,
       "Check home.html or pricing.html for developer earning info")

# Q4: Stripe/payment integration
stripe_refs = search_in_files(SRC, r'stripe|payment|billing|checkout', "*.py")
stripe_refs += search_in_files(JS_DIR, r'stripe|payment|billing|checkout', "*.js")
record("F5-Q4", "Payment/Stripe integration exists in codebase",
       len(stripe_refs) > 0,
       f"{len(stripe_refs)} files reference payment/Stripe")

# Q5: App pricing field in recipe/manifest schema
price_in_recipes = 0
for rp in recipes[:20]:
    try:
        data = json.loads(read_file(rp))
        if isinstance(data, dict) and ("price" in data or "pricing" in data or "tier" in data):
            price_in_recipes += 1
    except (json.JSONDecodeError, KeyError):
        pass
record("F5-Q5", "Apps/recipes support price field in schema",
       price_in_recipes > 0,
       f"{price_in_recipes}/20 sampled recipes have price field")

In [ ]:
# ============================================================
# Floor 6: API Consumer
# Persona: developer using Solace Browser APIs
# Questions probed:
#   Q1: Does /docs page exist with API reference?
#   Q2: Is there an interactive API explorer (Swagger/OpenAPI)?
#   Q3: Does the API docs show rate limits and auth requirements?
#   Q4: Are error responses consistent (structured JSON)?
#   Q5: Does the server expose a /docs or /openapi.json endpoint?
# ============================================================

# Q1: /docs page exists with API reference
docs_page = WEB / "docs.html"
if docs_page.exists():
    docs_content = read_file(docs_page)
    has_api_ref = bool(re.search(r'api|endpoint|request|response', docs_content, re.IGNORECASE))
    record("F6-Q1", "Documentation hub exists with API references",
           has_api_ref,
           f"API references found={has_api_ref}")
else:
    record("F6-Q1", "Documentation hub exists", False)

# Q2: Interactive API explorer (Swagger/OpenAPI)
swagger_found = False
# Check for OpenAPI/Swagger references in server code
server_files = list(SRC.rglob("*.py")) if SRC.exists() else []
server_files += [BROWSER_ROOT / "solace_browser_server.py"]
for sf in server_files:
    if sf.exists():
        content = read_file(sf)
        if re.search(r'swagger|openapi|/docs|fastapi|/openapi\.json', content, re.IGNORECASE):
            swagger_found = True
            break
# Also check web directory
for wf in [WEB / "docs.html", WEB / "docs" / "api.html"]:
    if wf.exists():
        content = read_file(wf)
        if re.search(r'swagger|openapi|interactive.*api|try.*endpoint', content, re.IGNORECASE):
            swagger_found = True
            break
record("F6-Q2", "Interactive API explorer (Swagger/OpenAPI) available",
       swagger_found)

# Q3: Rate limits and auth documented
rate_limit_doc = False
if docs_page.exists():
    content = read_file(docs_page)
    has_rate = bool(re.search(r'rate.?limit|throttl|quota', content, re.IGNORECASE))
    has_auth = bool(re.search(r'auth|token|api.?key|bearer', content, re.IGNORECASE))
    rate_limit_doc = has_rate or has_auth
# Also check MCP docs
mcp_doc = DOCS_DIR / "mcp.html"
if mcp_doc.exists():
    content = read_file(mcp_doc)
    if re.search(r'rate|auth|token', content, re.IGNORECASE):
        rate_limit_doc = True
record("F6-Q3", "API docs show rate limits or auth requirements",
       rate_limit_doc)

# Q4: Structured error responses in server
server_path = BROWSER_ROOT / "solace_browser_server.py"
if server_path.exists():
    server_code = read_file(server_path)
    # Look for consistent error response patterns
    error_responses = re.findall(r'json_response\(\s*\{.*?["\']error["\']', server_code, re.DOTALL)
    error_dict_responses = re.findall(r'["\']error["\']\s*:', server_code)
    record("F6-Q4", "Server returns structured error responses",
           len(error_dict_responses) > 3,
           f"{len(error_dict_responses)} structured error responses found")
else:
    record("F6-Q4", "Server structured error responses", False, "Server file not found")

# Q5: Server exposes /docs or /openapi.json endpoint
if server_path.exists():
    server_code = read_file(server_path)
    has_docs_endpoint = bool(re.search(r'/docs|/openapi|/api/v1', server_code))
    has_api_versioning = bool(re.search(r'/api/v\d', server_code))
    record("F6-Q5", "Server exposes API endpoints with versioning",
           has_docs_endpoint or has_api_versioning,
           f"/docs endpoint={has_docs_endpoint}, versioned API={has_api_versioning}")
else:
    record("F6-Q5", "Server API endpoints", False, "Server file not found")

In [ ]:
# ============================================================
# Floor 7: Theme Designer
# Persona: designer creating custom themes for the platform
# Questions probed:
#   Q1: Does the theme editor exist in settings?
#   Q2: Are CSS custom properties documented with ranges?
#   Q3: Does the theme system support dark/light auto-switching (prefers-color-scheme)?
#   Q4: Are there multiple built-in themes (dark, light, midnight)?
#   Q5: Can themes be exported/shared as JSON/CSS bundles?
# ============================================================

# Q1: Theme editor in settings
settings_path = WEB / "settings.html"
if settings_path.exists():
    settings_content = read_file(settings_path)
    has_theme_section = bool(re.search(r'appearance|theme.*editor|theme.*grid|theme-card|theme-choice',
                                        settings_content, re.IGNORECASE))
    has_theme_selector = bool(re.search(r'data-theme-choice', settings_content))
    record("F7-Q1", "Theme editor/selector exists in settings page",
           has_theme_section and has_theme_selector,
           f"theme_section={has_theme_section}, selector={has_theme_selector}")
else:
    record("F7-Q1", "Settings page exists", False)

# Q2: CSS custom properties documented
css_content = read_file(CSS_FILE) if CSS_FILE.exists() else ""
css_vars = re.findall(r'--(sb-[\w-]+)', css_content)
unique_vars = set(css_vars)
# Check for styleguide documentation
styleguide_path = WEB / "style-guide.html"
styleguide_exists = styleguide_path.exists()
# Check for theme system docs
theme_docs = list(BROWSER_ROOT.rglob("*theme*system*")) + list(BROWSER_ROOT.rglob("*styleguide*theme*"))
record("F7-Q2", "CSS custom properties defined (design tokens)",
       len(unique_vars) >= 20,
       f"{len(unique_vars)} unique --sb-* vars, style-guide page={styleguide_exists}, theme docs={len(theme_docs)}")

# Q3: Dark/light auto-switching via prefers-color-scheme
theme_js = JS_DIR / "theme.js"
if theme_js.exists():
    tj_content = read_file(theme_js)
    has_media_query = bool(re.search(r'prefers-color-scheme', tj_content))
    has_system_detect = bool(re.search(r'matchMedia|system.*pref|detect.*theme', tj_content, re.IGNORECASE))
    record("F7-Q3", "Theme auto-switches based on OS preference",
           has_media_query or has_system_detect,
           f"prefers-color-scheme={has_media_query}, matchMedia={has_system_detect}")
else:
    record("F7-Q3", "Theme auto-switch", False, "theme.js not found")

# Q4: Multiple built-in themes
theme_css_dir = WEB / "css" / "themes"
if theme_css_dir.exists():
    theme_files = list(theme_css_dir.glob("*.css"))
    theme_names = [f.stem for f in theme_files]
    record("F7-Q4", "Multiple built-in themes available",
           len(theme_files) >= 2,
           f"{len(theme_files)} themes: {theme_names}")
else:
    record("F7-Q4", "Theme CSS directory exists", False)

# Q5: Theme export/share capability
if theme_js.exists():
    tj_content = read_file(theme_js)
    has_export = bool(re.search(r'export|download|share|loadCustom|bundle', tj_content, re.IGNORECASE))
    has_custom_load = bool(re.search(r'loadCustom|customTheme|user.*theme', tj_content, re.IGNORECASE))
    record("F7-Q5", "Themes can be exported or custom-loaded",
           has_export or has_custom_load,
           f"export={has_export}, custom_load={has_custom_load}")
else:
    record("F7-Q5", "Theme export/load capability", False, "theme.js not found")

In [ ]:
# ============================================================
# Floor 8: Extension Builder
# Persona: developer building browser extensions/plugins
# Questions probed:
#   Q1: Is there a manifest.json schema for extensions?
#   Q2: Is the extension/plugin architecture documented?
#   Q3: Does the sealed store model limit extension capability clearly?
#   Q4: Can extensions hook into YinYang approval flow?
#   Q5: Does the security model prevent scope escalation?
# ============================================================

# Q1: Manifest.json schema for apps/extensions
manifest_files = list(BROWSER_ROOT.rglob("manifest*.json"))
# Exclude node_modules manifests
manifest_files = [m for m in manifest_files if "node_modules" not in str(m)]
# Check app data directory for manifest schemas
app_manifests = list((DATA / "apps").rglob("manifest*.json")) if (DATA / "apps").exists() else []
record("F8-Q1", "App/extension manifest schema exists",
       len(manifest_files) > 0 or len(app_manifests) > 0,
       f"{len(manifest_files)} manifest files, {len(app_manifests)} app manifests")

# Q2: Extension architecture documented
extension_doc_found = False
extension_doc_hits = []
for path in [WEB / "docs.html", BROWSER_ROOT / "CONTRIBUTING.md",
             DOCS_DIR / "quick-start.html"]:
    if path.exists():
        content = read_file(path)
        if re.search(r'extension|plugin|hook|lifecycle|install.*event|uninstall.*event',
                     content, re.IGNORECASE):
            extension_doc_found = True
            extension_doc_hits.append(str(path.relative_to(BROWSER_ROOT)))
# Also check src for extension architecture
ext_arch = search_in_files(SRC, r'extension|plugin.*arch|app.*lifecycle', "*.py")
record("F8-Q2", "Extension/plugin architecture documented or implemented",
       extension_doc_found or len(ext_arch) > 0,
       f"Docs: {extension_doc_hits}, Code: {ext_arch[:3]}")

# Q3: Sealed store limits documented clearly
sealed_doc = False
for path in [WEB / "app-store.html", WEB / "docs.html", BROWSER_ROOT / "CONTRIBUTING.md"]:
    if path.exists():
        content = read_file(path)
        if re.search(r'sealed.*store|no.*plugin|users?.*suggest.*we.*implement|cannot.*fork',
                     content, re.IGNORECASE):
            sealed_doc = True
            break
record("F8-Q3", "Sealed store limits clearly documented", sealed_doc)

# Q4: YinYang approval flow hooks
yinyang_files = list(JS_DIR.glob("yinyang-*.js"))
yinyang_hook_capable = False
for yf in yinyang_files:
    content = read_file(yf)
    if re.search(r'hook|custom.*approv|extensi.*approv|app.*hook', content, re.IGNORECASE):
        yinyang_hook_capable = True
        break
record("F8-Q4", "YinYang approval flow supports extension hooks",
       yinyang_hook_capable,
       f"{len(yinyang_files)} yinyang JS files checked")

# Q5: Security model prevents scope escalation
scope_guard_files = search_in_files(SRC, r'scope.*guard|scope.*check|escalat|permission.*check', "*.py")
scope_guard_js = search_in_files(JS_DIR, r'scope.*guard|scope.*check|permission', "*.js")
record("F8-Q5", "Security model has scope guarding/escalation prevention",
       len(scope_guard_files) > 0 or len(scope_guard_js) > 0,
       f"Python: {scope_guard_files[:3]}, JS: {scope_guard_js[:3]}")

In [ ]:
# ============================================================
# Floor 9: Documentation Writer
# Persona: evaluating developer documentation quality
# Questions probed:
#   Q1: Does every public docs page have title, description, code examples?
#   Q2: Is the quick-start guide completable in under 10 minutes?
#   Q3: Are there architecture diagrams?
#   Q4: Do docs have table of contents navigation?
#   Q5: Does docs have search functionality?
# ============================================================

# Q1: Docs pages have title, description, code examples
docs_pages = list(DOCS_DIR.glob("*.html")) if DOCS_DIR.exists() else []
docs_quality = {}
for dp in docs_pages:
    content = read_file(dp)
    has_title = bool(re.search(r'<h1|<title', content, re.IGNORECASE))
    has_desc = bool(re.search(r'<meta.*description|<p.*class.*lead', content, re.IGNORECASE))
    has_code = bool(re.search(r'<code|<pre', content, re.IGNORECASE))
    docs_quality[dp.name] = {"title": has_title, "desc": has_desc, "code": has_code}

complete_docs = sum(1 for d in docs_quality.values() if all(d.values()))
record("F9-Q1", "Docs pages have title + description + code examples",
       complete_docs >= len(docs_pages) * 0.5 if docs_pages else False,
       f"{complete_docs}/{len(docs_pages)} complete: {docs_quality}")

# Q2: Quick-start guide exists and is concise
qs_path = DOCS_DIR / "quick-start.html"
if qs_path.exists():
    qs_content = read_file(qs_path)
    # Check for step-by-step structure
    steps = re.findall(r'step|Step|\d\.\s', qs_content)
    # Check for time estimate
    has_time = bool(re.search(r'\d+\s*minute|5.?min|quick', qs_content, re.IGNORECASE))
    # Check for install instructions
    has_install = bool(re.search(r'install|pip|npm|download|clone', qs_content, re.IGNORECASE))
    record("F9-Q2", "Quick-start guide is step-by-step and fast",
           len(steps) >= 3 and has_install,
           f"{len(steps)} step references, time_estimate={has_time}, install={has_install}")
else:
    record("F9-Q2", "Quick-start guide exists", False)

# Q3: Architecture diagrams
# Check for mermaid diagrams in docs
diagram_hits = search_in_files(DOCS_DIR, r'mermaid|diagram|flowchart|sequenceDiagram|graph', "*.html")
# Check src/diagrams
diagram_dir = SRC / "diagrams" if SRC.exists() else None
diagram_count = count_files(diagram_dir, "*.mmd") + count_files(diagram_dir, "*.mermaid") if diagram_dir else 0
record("F9-Q3", "Architecture diagrams available",
       len(diagram_hits) > 0 or diagram_count > 0,
       f"Docs with diagrams: {diagram_hits[:3]}, diagram files: {diagram_count}")

# Q4: Table of contents in docs
toc_found = 0
for dp in docs_pages:
    content = read_file(dp)
    if re.search(r'docs-toc|table.*content|on.*this.*page', content, re.IGNORECASE):
        toc_found += 1
record("F9-Q4", "Docs pages have table of contents",
       toc_found >= len(docs_pages) * 0.5 if docs_pages else False,
       f"{toc_found}/{len(docs_pages)} pages have TOC")

# Q5: Docs search functionality
if docs_page.exists():
    docs_hub_content = read_file(docs_page)
    has_search = bool(re.search(r'search|filter|<input.*search', docs_hub_content, re.IGNORECASE))
    record("F9-Q5", "Documentation has search functionality", has_search)
else:
    record("F9-Q5", "Docs search", False)

In [ ]:
# ============================================================
# Floor 10: Community Contributor
# Persona: open-source contributor wanting to help build the ecosystem
# Questions probed:
#   Q1: Does CONTRIBUTING.md exist?
#   Q2: Does CONTRIBUTING.md explain how to contribute recipes?
#   Q3: Does the FSL license make community rights clear?
#   Q4: Is there a community forum/Discord/discussion link?
#   Q5: Is there a contributor code of conduct?
# ============================================================

# Q1: CONTRIBUTING.md exists
contributing_path = BROWSER_ROOT / "CONTRIBUTING.md"
record("F10-Q1", "CONTRIBUTING.md exists",
       file_exists(contributing_path),
       f"Path: {contributing_path}")

# Q2: CONTRIBUTING.md explains recipe contribution
if contributing_path.exists():
    contrib_content = read_file(contributing_path)
    has_recipe_guide = bool(re.search(r'recipe', contrib_content, re.IGNORECASE))
    has_pr_guide = bool(re.search(r'pull.*request|PR|fork', contrib_content, re.IGNORECASE))
    has_bug_guide = bool(re.search(r'bug|issue', contrib_content, re.IGNORECASE))
    record("F10-Q2", "CONTRIBUTING.md covers recipes, PRs, and bugs",
           has_recipe_guide and has_pr_guide,
           f"recipes={has_recipe_guide}, PRs={has_pr_guide}, bugs={has_bug_guide}")
else:
    record("F10-Q2", "CONTRIBUTING.md covers contribution types", False, "File does not exist")

# Q3: License file exists and FSL explained
license_path = BROWSER_ROOT / "LICENSE"
license_md = BROWSER_ROOT / "LICENSE.md"
has_license = file_exists(license_path) or file_exists(license_md)
fsl_explained = False
for lp in [license_path, license_md]:
    if lp.exists():
        content = read_file(lp)
        if re.search(r'FSL|Functional Source|source.?available', content, re.IGNORECASE):
            fsl_explained = True
            break
# Also check CONTRIBUTING.md
if contributing_path.exists():
    content = read_file(contributing_path)
    if re.search(r'FSL|Functional Source|license', content, re.IGNORECASE):
        fsl_explained = True
record("F10-Q3", "License (FSL) is present and explained",
       has_license or fsl_explained,
       f"LICENSE file={has_license}, FSL mentioned={fsl_explained}")

# Q4: Community links (Discord, forum, discussions)
community_link_found = False
community_channels = []
for path in [contributing_path, BROWSER_ROOT / "README.md", WEB / "home.html", WEB / "docs.html"]:
    if path.exists():
        content = read_file(path)
        for term in ['discord', 'forum', 'discussion', 'community', 'slack', 'chat']:
            if re.search(term, content, re.IGNORECASE):
                community_link_found = True
                community_channels.append(f"{path.name}:{term}")
record("F10-Q4", "Community forum/Discord/discussion linked",
       community_link_found,
       f"Found: {community_channels[:5]}" if community_channels else "No community links")

# Q5: Code of conduct
coc_paths = [
    BROWSER_ROOT / "CODE_OF_CONDUCT.md",
    BROWSER_ROOT / "code_of_conduct.md",
    BROWSER_ROOT / ".github" / "CODE_OF_CONDUCT.md",
]
has_coc = any(file_exists(p) for p in coc_paths)
# Check if CONTRIBUTING.md references conduct
coc_in_contrib = False
if contributing_path.exists():
    content = read_file(contributing_path)
    coc_in_contrib = bool(re.search(r'conduct|respect|guideline|behav', content, re.IGNORECASE))
record("F10-Q5", "Code of conduct or community guidelines exist",
       has_coc or coc_in_contrib,
       f"COC file={has_coc}, in CONTRIBUTING={coc_in_contrib}")

In [ ]:
# ============================================================
# Floor 11: Accessibility-First App Maker
# Persona: developer building accessible apps on the platform
# Questions probed:
#   Q1: Do platform HTML pages include ARIA attributes and semantic HTML?
#   Q2: Does the CSS provide accessible component patterns?
#   Q3: Are there accessibility testing tools/checklists for app makers?
#   Q4: Does the theme system ensure contrast compliance?
#   Q5: Do pages support keyboard navigation (focus indicators, skip-nav)?
# ============================================================

# Q1: ARIA attributes and semantic HTML across pages
html_files = list(WEB.glob("*.html"))
aria_counts = {}
semantic_counts = {}
for hf in html_files:
    content = read_file(hf)
    aria_count = len(re.findall(r'aria-', content))
    semantic_count = len(re.findall(r'<(main|nav|header|footer|section|article|aside|figure|figcaption)\b',
                                      content, re.IGNORECASE))
    if aria_count > 0 or semantic_count > 0:
        aria_counts[hf.name] = aria_count
        semantic_counts[hf.name] = semantic_count

total_aria = sum(aria_counts.values())
total_semantic = sum(semantic_counts.values())
pages_with_aria = len([v for v in aria_counts.values() if v > 0])
record("F11-Q1", "Pages use ARIA attributes and semantic HTML",
       total_aria > 10 and total_semantic > 10,
       f"{total_aria} aria attrs across {pages_with_aria} pages, {total_semantic} semantic elements")

# Q2: Accessible component patterns in CSS
css_content = read_file(CSS_FILE) if CSS_FILE.exists() else ""
has_sr_only = bool(re.search(r'sr-only|visually-hidden|screen-reader', css_content, re.IGNORECASE))
has_focus_styles = bool(re.search(r':focus|:focus-visible|:focus-within', css_content))
has_contrast = bool(re.search(r'contrast|prefers-contrast', css_content, re.IGNORECASE))
record("F11-Q2", "CSS provides accessible component patterns",
       has_sr_only and has_focus_styles,
       f"sr-only={has_sr_only}, focus_styles={has_focus_styles}, contrast_support={has_contrast}")

# Q3: Accessibility testing tools or checklists
a11y_tools = search_in_files(BROWSER_ROOT / "scripts", r'a11y|access|axe|lighthouse|wcag', "*.py")
a11y_tools += search_in_files(BROWSER_ROOT / "tests", r'a11y|access|wcag', "*.py")
# Check docs for a11y guidelines
a11y_docs = search_in_files(DOCS_DIR, r'access|a11y|wcag|screen.?reader', "*.html")
record("F11-Q3", "Accessibility testing tools or guidelines available",
       len(a11y_tools) > 0 or len(a11y_docs) > 0,
       f"Tools: {a11y_tools[:3]}, Docs: {a11y_docs[:3]}")

# Q4: Theme system ensures contrast compliance
for theme_file in (WEB / "css" / "themes").glob("*.css") if (WEB / "css" / "themes").exists() else []:
    content = read_file(theme_file)
    # Light theme is the one most likely to have contrast issues
    if theme_file.stem == "light":
        # Check that text colors are dark enough against light background
        has_dark_text = bool(re.search(r'--sb-text:\s*#[0-3]', content))
        record("F11-Q4", "Light theme has high-contrast text colors",
               has_dark_text,
               f"Dark text on light bg={has_dark_text}")
        break
else:
    record("F11-Q4", "Theme contrast compliance", False, "No theme files found")

# Q5: Keyboard navigation support
skip_nav_found = False
for hf in html_files:
    content = read_file(hf)
    if re.search(r'skip.*nav|skip.*content|skip.*main', content, re.IGNORECASE):
        skip_nav_found = True
        break
# Check JS for skip-nav injection
for jf in JS_DIR.glob("*.js"):
    content = read_file(jf)
    if re.search(r'skip.*nav|skipnav|skip-nav', content, re.IGNORECASE):
        skip_nav_found = True
        break

# Focus management
focus_trap = search_in_files(JS_DIR, r'focus.?trap|tab.?index|focusable', "*.js")
record("F11-Q5", "Keyboard navigation with skip-nav and focus management",
       skip_nav_found,
       f"skip-nav={skip_nav_found}, focus_management={len(focus_trap)} files")

In [ ]:
# ============================================================
# Floor 12: i18n-Aware App Maker
# Persona: developer building internationalized apps
# Questions probed:
#   Q1: Does the platform use a t() or data-i18n translation system?
#   Q2: Are locale files available for multiple languages?
#   Q3: Does the CSS support RTL layout?
#   Q4: Can apps declare supported locales?
#   Q5: Does the i18n system handle pluralization or complex forms?
# ============================================================

# Q1: Translation system (data-i18n or t() function)
i18n_html_hits = search_in_files(WEB, r'data-i18n', "*.html")
i18n_js_hits = search_in_files(JS_DIR, r'data-i18n|t\(|translate|i18n', "*.js")
total_i18n_tags = sum(count for _, count in i18n_html_hits)
record("F12-Q1", "Platform uses i18n translation system (data-i18n)",
       total_i18n_tags > 20,
       f"{total_i18n_tags} data-i18n tags across {len(i18n_html_hits)} HTML files, {len(i18n_js_hits)} JS files with i18n")

# Q2: Locale files for multiple languages
locale_coverage = {}
if LOCALES_DIR.exists():
    for locale_subdir in LOCALES_DIR.iterdir():
        if locale_subdir.is_dir():
            json_files = list(locale_subdir.glob("*.json"))
            total_keys = 0
            for jf in json_files:
                try:
                    data = json.loads(read_file(jf))
                    if isinstance(data, dict):
                        total_keys += len(data)
                except (json.JSONDecodeError, ValueError):
                    pass
            locale_coverage[locale_subdir.name] = {"files": len(json_files), "keys": total_keys}

languages_with_content = [k for k, v in locale_coverage.items() if v["keys"] >= 5]
record("F12-Q2", "Locale files available for multiple languages",
       len(languages_with_content) >= 5,
       f"{len(languages_with_content)} languages with 5+ keys: {sorted(languages_with_content)}")

# Q3: RTL CSS support
css_content = read_file(CSS_FILE) if CSS_FILE.exists() else ""
rtl_rules = re.findall(r'\[dir=.?rtl.?\]', css_content)
rtl_in_css = len(rtl_rules)

# Check for RTL-specific CSS file
schedule_css = read_file(WEB / "css" / "schedule.css") if (WEB / "css" / "schedule.css").exists() else ""
rtl_in_schedule = len(re.findall(r'\[dir=.?rtl.?\]', schedule_css))

# Check HTML for dir attribute support
dir_attr_hits = search_in_files(WEB, r'dir=|direction.*rtl|\[dir', "*.html")
record("F12-Q3", "CSS supports RTL layout",
       rtl_in_css > 0 or rtl_in_schedule > 0,
       f"{rtl_in_css} RTL rules in site.css, {rtl_in_schedule} in schedule.css, dir attr in {len(dir_attr_hits)} pages")

# Q4: Apps can declare supported locales (in recipe/manifest)
locale_in_recipes = 0
for rp in recipes[:20]:
    try:
        data = json.loads(read_file(rp))
        if isinstance(data, dict):
            if any(k in data for k in ["locales", "supported_locales", "languages", "locale"]):
                locale_in_recipes += 1
    except (json.JSONDecodeError, KeyError):
        pass
record("F12-Q4", "Apps/recipes can declare supported locales",
       locale_in_recipes > 0,
       f"{locale_in_recipes}/20 sampled recipes have locale declaration")

# Q5: Pluralization support
plural_support = False
for jf in JS_DIR.glob("*.js"):
    content = read_file(jf)
    if re.search(r'plural|Intl\.PluralRules|pluralize|_n\(|ngettext', content, re.IGNORECASE):
        plural_support = True
        break
# Also check locale files for plural forms
for locale_dir in (LOCALES_DIR.iterdir() if LOCALES_DIR.exists() else []):
    if locale_dir.is_dir():
        for jf in locale_dir.glob("*.json"):
            content = read_file(jf)
            if re.search(r'plural|_one|_many|_few|_other', content, re.IGNORECASE):
                plural_support = True
                break
record("F12-Q5", "i18n system handles pluralization",
       plural_support,
       "Pluralization support found" if plural_support else "No pluralization patterns detected")

In [ ]:
# ============================================================
# EVIDENCE SUMMARY -- Tower 15: App Makers x Solace Browser
# ============================================================

passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
finding_rate = (failed / total * 100) if total > 0 else 0

# Group by floor
floor_summary = {}
for r in results:
    floor = r["id"].split("-")[0]  # e.g. "F1"
    if floor not in floor_summary:
        floor_summary[floor] = {"passed": 0, "failed": 0, "total": 0}
    floor_summary[floor]["total"] += 1
    if r["status"] == "PASS":
        floor_summary[floor]["passed"] += 1
    else:
        floor_summary[floor]["failed"] += 1

floor_labels = {
    "F1": "First-Time Builder", "F2": "Recipe Author", "F3": "OAuth3 Integrator",
    "F4": "Store Publisher", "F5": "Revenue Developer", "F6": "API Consumer",
    "F7": "Theme Designer", "F8": "Extension Builder", "F9": "Documentation Writer",
    "F10": "Community Contributor", "F11": "Accessibility App Maker", "F12": "i18n App Maker",
}

summary = {
    "surface": NORTHSTAR,
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total,
    "passed": passed,
    "failed": failed,
    "finding_rate": round(finding_rate, 1),
    "score": round(passed / total * 100, 1) if total > 0 else 0,
    "min_score": MIN_SCORE,
    "converged": finding_rate < 5,
    "floor_scores": {
        f"{k} ({floor_labels.get(k, '?')})": f"{v['passed']}/{v['total']}"
        for k, v in sorted(floor_summary.items())
    },
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 70)
print(f"TOWER {TOWER}: {TOWER_NAME} -- SOLACE BROWSER QA")
print("=" * 70)
print(f"Total probes:  {total}")
print(f"Passed:        {passed}")
print(f"Failed:        {failed}")
print(f"Score:         {summary['score']}%")
print(f"Finding rate:  {finding_rate:.1f}%")
print(f"Converged:     {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
print()
print("FLOOR BREAKDOWN:")
for k, v in sorted(floor_summary.items()):
    label = floor_labels.get(k, "?")
    pct = round(v["passed"] / v["total"] * 100) if v["total"] > 0 else 0
    bar = "#" * (pct // 5) + "." * (20 - pct // 5)
    print(f"  {k:4s} {label:28s} {v['passed']}/{v['total']} [{bar}] {pct}%")

if summary["findings"]:
    print()
    print("FINDINGS (FAIL):")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")

print()
print(json.dumps(summary, indent=2))